# Tutorial 06 — Minimum Squared Euclidean Distance and the Union Bound

The high-SNR slope of a coded BER curve is controlled by the *minimum squared Euclidean distance* (MSED) of the error events. For the rate-2 NSM trellis we expose three definitions in `nsm.bounds`:

| Function | Counts | Use |
|---|---|---|
| `min_squared_distance_stream0` | Error events that disturb stream-0 only | **Paper-matching**. Unbalanced > balanced. |
| `min_squared_distance_trellis` | Events with stream-0 disturbance, allowing stream-1 differences | Combined trellis. |
| `min_squared_distance`         | All events (incl. parallel single-bit `b_1` flips) | Full joint detector. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nsm.modem.msprs import precompute
from nsm.bounds import (min_squared_distance, min_squared_distance_trellis,
                        min_squared_distance_stream0, union_bound_ber)

## Compare metrics across L0 and filter type

In [ ]:
rows = []
for L0 in (3, 4, 5, 6):
    for ft in ('balanced', 'unbalanced'):
        p = precompute(L0, 1000, ft)
        rows.append((L0, ft, min_squared_distance(p),
                     min_squared_distance_trellis(p),
                     min_squared_distance_stream0(p)))
print(f'{"L0":>3} {"type":>11} {"d²_min":>10} {"d²_tre":>10} {"d²_str0":>10}')
for r in rows: print(f'{r[0]:>3} {r[1]:>11} {r[2]:>10.3f} {r[3]:>10.3f} {r[4]:>10.3f}')

## Overlay union bound on simulated uncoded BER

In [ ]:
from nsm.modem.msprs import uncoded_ber
from nsm.channel.awgn import setup
L0, src = 3, 4096
p = precompute(L0, src, 'unbalanced')
snrs = np.arange(2, 18.1, 2.0); ber = []
for snr in snrs:
    nv = 0.5 / (2 * 10 ** (snr / 10))
    total = sum(int(uncoded_ber(src, L0, p['h0'], p['h1'], p['branch_labels'],
                                p['memory'], p['total_states'], p['next_states'],
                                p['branch_indices'], nv, np.sqrt(nv)))
                 for _ in range(3))
    ber.append(total / (3 * src))
d2 = min_squared_distance_stream0(p)
bound = union_bound_ber(d2, snrs, rate=1.0, bits_per_symbol=2.0)
plt.semilogy(snrs, np.clip(ber, 1e-6, 1), 'o-', label='simulation')
plt.semilogy(snrs, bound, 'k--', label=f'union bound, d²_str0={d2:.2f}')
plt.xlabel('Eb/N0 (dB)'); plt.ylabel('BER'); plt.grid(True, which='both', alpha=0.3)
plt.legend(); plt.title('Uncoded MS-PRS L0=3 unbalanced vs union bound')